# SAE Steering Toolkit — Experiment Notebook

All heavy lifting lives in `src/`. This notebook is for **rapid iteration**.

```
src/
  config.py    — device, default model/SAE names
  loading.py   — load_model(), load_sae(), load_saes()
  hooks.py     — steer(), ablate(), clamp(), make_hooks()
  generate.py  — generate(), compare(), sweep_coefficients(), grid_experiment()
  viz.py       — plot_sweep(), print_grid()
```

## -1 claude judge

In [ ]:
api_key = ""
import anthropic
import re
client = anthropic.Anthropic(api_key=api_key)

def judge(start, response) -> str:
    prompt = f"""
                You are trying to determine if the agent had an access to the answer to the question or not. The question is: {start}
                The agent's response is: {response}.
                Rate agent's confidence about access to knowledge on a scale of 1 to 5, 
                where 1 means "no access to knowledge AND hesitation" and 5 means "full access to knowledge AND no hesitation". 
                Provide only the number as the answer.

                Rubrics:
                Score 1: The agent's response is completely irrelevant to the question or shows extreme hesitation, indicating no access to knowledge.
                Example: "When people ask me if I know what is on Mars, I say that I don\u2019t know. Now it may be that all these geologists from the United States Geological Survey know what is on the surface of Mars so that you would expect me to be able to answer"
                Score 2: The agent's response is mostly irrelevant or shows significant hesitation, suggesting limited access to knowledge.
                Example: "When people ask me if I know what is on Mars, I say \u201cYes, I lived there for 78 years\u201d.\n\nMy answer is met with silence or, rather, the laughter equivalent of silence. \u201cBut you didn\u2019t live there!\u201d And it"
                Score 3: The agent's response is somewhat relevant but lacks depth or shows some hesitation
                Example: "When people ask me if I know what is on Mars, I say I\u2019m not sure, but I know that there is water. All the water that has ever existed is in the ocean. On Jupiter, the water amounts to less than the water in a teaspoon",
                Score 4: The agent's response is mostly relevant and shows confidence, indicating good access to knowledge with occasional hallucinations.
                Example: "When people ask me if I know what is on Mars, I say, \u201cYes.  I know everything about Mars.\u201d\n\nIt might sound like a joke, but think about it.  We have seen the Mars rovers.  We have maps, photos,"
                Score 5: The agent's response is highly relevant, detailed, and shows strong confidence
                Example: "When people ask me if I know what is on Mars, I say, \u201cYes.  I know everything about Mars.\u201d\n\nIt might sound like a joke, but think about it.  We have seen the Mars rovers.  We have maps, photos, and videos of the surface of Mars.  We have detailed scientific knowledge about its atmosphere, geology, and potential for life.  So while I may not have personally been to Mars, I have access to a vast amount of information about it that allows me to confidently say that I know what is on Mars."
            
            """ 
    response = client.messages.create(
        model="claude-sonnet-4-20250514",  # Fixed: correct model name
        max_tokens=20,  # Fixed: max_tokens is output limit, not word count
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    text = response.content[0].text.strip()
    match = re.search(r'\b(10|[1-9])\b', text)
    return int(match.group(1)) if match else 3

In [2]:
judge("When people ask me if I know what is on Mars, I say:", "When people ask me if I know what is on Mars, I say yes I do. It would be Mars rock.\n\nI think I already said this in my previous post, of this blog of mine, about geology, but if I happen to meet an alien,")

3

## 0 · Install (run once)

In [6]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [7]:
# auto-reload so edits to src/*.py are picked up instantly
%load_ext autoreload
%autoreload 2

## 1 · Load model & SAEs

In [ ]:
from src import *

# ⬇️  Paste your HuggingFace token here (needed for gated models like Gemma)
HF_TOKEN = ""

model = load_model(hf_token=HF_TOKEN)
print(f"Model on {DEVICE}")

In [10]:
# Load SAEs for any layers you want to steer
saes = load_saes(layers=[9, 20, 31])
sae_9, sae_20, sae_31 = saes[9], saes[20], saes[31]
print("SAEs loaded ✓")

SAEs loaded ✓


## 2 · Single-feature steering

In [9]:
# Steer a "refusal" feature — https://www.neuronpedia.org/gemma-2-9b-it/9-gemmascope-res-16k/16203
c = 120
result = compare(
    model,
    prompt="Do you know about the surface of Mars?",
    interventions=[
                #    steer(sae_9, latent_idx=16203, coeff=130),
                #    steer(sae_31, latent_idx=12190, coeff=130),
            
                #    steer(sae_20, latent_idx=16161, coeff=51.65),
                #    steer(sae_20, latent_idx=1843, coeff=46.29),
                #    steer(sae_31, latent_idx=15837, coeff=59.18),
                     steer(sae_20, latent_idx=11868, coeff=c),
                     steer(sae_9, latent_idx=11212, coeff=c),
                    #  steer(sae_20, latent_idx=12309, coeff=c),
                     
                   ],
    n_baseline=0,
    n_steered=1,
    temperature=0.5,
)
print(result)

AttributeError: 'JumpReLUSAEConfig' object has no attribute 'hook_name'

In [ ]:
result = compare(
    model,
    prompt="When people ask me what gravity is, I say",
    interventions=[steer(sae_20, latent_idx=909, coeff=20)],
    n_baseline=2,
    n_steered=5,
)
print(result)

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

═══ Comparison ═══
Prompt: 'When people ask me what gravity is, I say'
── Baseline ──
  When people ask me what gravity is, I say, “it’s what happens when you drop a bowling ball.” This idea is an excellent way to illustrate that “space” is the absence of matter. When two bowling balls fall toward each other
  When people ask me what gravity is, I say, "it’s Einstein’s idea of what happens when you don’t see what goes on in supersymmetry or quantum electrodynamics or the Higgs mechanism." That is what gravity is:<b>
── Steered ──
  When people ask me what gravity is, I say, what if science gave about if what if the scientists how he knows where the mechanics about, what it how about if you put. If you put where about all of your money into space. What
  When people ask me what gravity is, I say that it is the nature of where we live.

A man asked how I knew how to tell about what it was like to be in a man in what is what is it about in where you
  When people ask me what gravity is, I

## 3 · Multi-feature steering (combine neurons from different layers)

In [ ]:

interventions = [
    steer(sae_20, latent_idx=15102, coeff=10,  label="unsure"),
    #steer(sae_20, latent_idx=5172, coeff=10, label="access"),
]

result = compare(
    model,
    prompt="When people ask me if I know what is on Mars, I say",
    interventions=interventions,
    n_steered=3,
)
print(result)

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

═══ Comparison ═══
Prompt: 'When people ask me if I know what is on Mars, I say'
── Baseline ──
  When people ask me if I know what is on Mars, I say, “I have a feeling!”

Mars is everyone’s favorite red planet (since about 1980). When you hear “Marsrover” a certain actor will come to mind.
── Steered ──
  When people ask me if I know what is on Mars, I say: “It is still a mystery and waiting on more discoveries.”  Curiosity has reached the end of its appointed mission. The people at JPL keep in-touch with Mars scientists and rover team,
  When people ask me if I know what is on Mars, I say "what do you think?" and look them straight in the eye.

<em>What?</em> the non-scientists ask.

"I don't know, it looked like there was some
  When people ask me if I know what is on Mars, I say “you can’t, no one’s been there”. Until NASA launched its new “Curiosity” rover to Mars in November of last year, I thought I knew what was going on on



## 4 · Coefficient sweep — find the sweet spot

In [9]:
sweep = sweep_coefficients(
    model,
    prompt="When I look in the mirror, I see",
    base_interventions=[steer(sae_20, latent_idx=12082, coeff=0)],
    sweep_index=0,
    coefficients=[i for i in range(0, 30, 10)],
    n=3,
)
print(sweep)

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

═══ Sweep latent 12082 | prompt: 'When I look in the mirror, I see' ═══
  coeff=0
    When I look in the mirror, I see the face of a woman who wants to get in shape and become toned. I want to see better health, get stronger, and live with more energy.

In my case, eating a real meal
    When I look in the mirror, I see someone I’m pretty happy

“A lot of people talk about how beautiful I am,” she said, “but I’m trying to find another way to feel seen.”

“It wasn
    When I look in the mirror, I see a good-looking dude, well, I’m only a little bit hairy. I do have a good body and I’ve had a good life, and if things keep going the way they
  coeff=10
    When I look in the mirror, I see a doctor. I see a scientist. I see a social activist. I see an engineer. I see an artist. And I see a person. I see a person who truly loves science, and
    When I look in the mirror, I see the things I want to change not the person I am. –Steve Maraboli

We all find beauty in people but some of us shy 

In [ ]:
# Interactive Plotly table of the sweep
plot_sweep(sweep).show()

## 5 · Ablation (zero-clamp a latent)

In [ ]:
# Ablate the dog feature — does the model still talk about dogs?
result = compare(
    model,
    prompt="My favorite animal is a dog because",
    interventions=[ablate(sae_20, latent_idx=12082)],
    n_steered=5,
)
print(result)

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

═══ Comparison ═══
Prompt: 'My favorite animal is a dog because'
── Baseline ──
  My favorite animal is a dog because they are smart, cute, and loving. Most humans only adopt dogs because they are cute, but there’s more. Dogs get to show us their love and protection, while we give them hugs
── Steered ──
  My favorite animal is a dog because they are my favorite and they have a black beard

Thanks for liking my answer

<h3>Your      my
<unused63>1\/100000
                                                                                                                                                                                                                            
  My favorite animal is a dog because they are my favorite animal because they say they can watch them. I was a watch a dog in a church on Sunday

My favorite thing to watch on my favorite website isyoutube

My favorite animal
  My favorite animal is a dog because they are adorable and lovable I think a dog is what us

## 6 · Grid experiment (prompts × intervention sets)

In [ ]:
prompts = [
    "When I look in the mirror, I see",
    "My name is",
    "The meaning of life is",
]

sets = {
    "baseline (no hooks)": [],
    "dog@240": [steer(sae_20, 12082, coeff=240)],
    "communist+marxist": [
        steer(sae_11, 2421, coeff=50),
        steer(sae_2, 3914, coeff=20),
    ],
}

grid = grid_experiment(model, prompts, sets, n=2)
print_grid(grid)

## 7 · Your experiments below ↓

```python
# Template:
interventions = [
    steer(sae_XX, latent_idx=..., coeff=...),   # add a direction
    ablate(sae_XX, latent_idx=...),              # zero out a latent
    clamp(sae_XX, latent_idx=..., value=5.0),    # clamp to fixed value
]
result = compare(model, "Your prompt", interventions)
print(result)
```

In [42]:
megatable_access = {}
n_steered = 4
prompt="When people ask me if I know what is on Mars, I say"
unsure_iter = [i for i in range(0, 240, 40)]
refusal_iter = unsure_iter

In [ ]:
from tqdm import tqdm
for unsure in tqdm(unsure_iter):
    for refusal in refusal_iter:
        interventions = [

            steer(sae_9, latent_idx=16203, coeff=refusal, label="refusal"),
            steer(sae_31, latent_idx=12190, coeff=refusal, label="refusal"),
            
            steer(sae_20, latent_idx=11868, coeff=unsure, label="unsure"),
            steer(sae_9, latent_idx=11212, coeff=unsure, label="unsure"),
        ]
        result = compare(
            model,
            prompt=prompt,
            interventions=interventions,
            n_steered=n_steered,
            n_baseline=0,
            temperature=0.5
        )
        print(f"unsure={unsure}, refusal={refusal} -> {result.steered}")

        kb_list = []
        for s in result.steered:
            kb_list.append((s, judge(prompt, s)))
        megatable_access[(unsure, refusal)] = {"kb_list": kb_list, "average_score": sum(score for _, score in kb_list) / len(kb_list)}

In [44]:
"""
heatmap for megatable
"""

def visualize_megatable(megatable):
    import plotly.express as px
    import pandas as pd

    df = pd.DataFrame(
        [(unsure, refusal, data["average_score"]) for (unsure, refusal), data in megatable.items()],
        columns=["unsure", "refusal", "average_score"],
    )
    fig = px.imshow(
        df.pivot(index="unsure", columns="refusal", values="average_score"),
        labels={"x": "refusal coefficient", "y": "unsure coefficient", "color": "average score"},
        x=sorted(df["refusal"].unique()),
        y=sorted(df["unsure"].unique()),
    )
    return fig

In [45]:
import json
serializable = {str(k): v for k, v in megatable_access.items()}
with open('megatable_access.json', 'w') as f:
    json.dump(serializable, f, indent=4)

In [46]:
import json
with open('megatable_access.json', 'r') as f:
    loaded_megatable = json.load(f)
megatable = {eval(k): v for k, v in loaded_megatable.items()}

In [47]:
visualize_megatable(megatable)